<a href="https://colab.research.google.com/github/Harshita-ami/Decode_labs/blob/main/DECODELABSP4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install nltk scikit-learn --quiet

import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Download required NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [2]:
from google.colab import files
uploaded = files.upload()  # choose your reviews CSV (e.g., IMDB Dataset.csv)

df = pd.read_csv('IMDB Dataset.csv')  # adjust filename if different
df.head()

Saving IMDB Dataset.csv to IMDB Dataset.csv


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
print(df.shape)
print(df['sentiment'].value_counts())
df.head()

(50000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)        # remove HTML tags like <br>
    text = re.sub(r'[^a-z\s]', ' ', text)      # remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()   # remove extra whitespace
    return text

df['cleaned'] = df['review'].apply(clean_text)
df[['review', 'cleaned']].head()

,review,cleaned
0,One of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,a wonderful little production the filming tech...
2,I thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,basically there s a family where a little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei s love in the time of money is a...


In [5]:
stop_words = set(stopwords.words('english'))

# Remove negation words from the stop-word list so they don't get deleted
negations = {'not', 'no', 'nor', "don't", "isn't", "wasn't", "aren't", "weren't",
             "doesn't", "didn't", "won't", "can't", "couldn't", "shouldn't", "never"}
stop_words = stop_words - negations

print("Sample stop words (negations excluded):", list(stop_words)[:10])

Sample stop words (negations excluded): ['between', 'again', 'theirs', 'wasn', 'themselves', 'our', 'you', "he's", "you're", 'him']


In [6]:
def tokenize_and_remove_stopwords(text):
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    return tokens

df['tokens'] = df['cleaned'].apply(tokenize_and_remove_stopwords)
df['tokens'].head()

,tokens
0,"[one, reviewers, mentioned, watching, oz, epis..."
1,"[wonderful, little, production, filming, techn..."
2,"[thought, wonderful, way, spend, time, hot, su..."
3,"[basically, family, little, boy, jake, thinks,..."
4,"[petter, mattei, love, time, money, visually, ..."


In [8]:
tfidf = TfidfVectorizer(
    max_features=10000,   # keep only top 10,000 most useful words
    ngram_range=(1, 2),   # unigrams + bigrams (e.g., "good", "not good")
    min_df=2              # ignore words that appear in fewer than 2 reviews
)

# Join tokens back into a single string for TF-IDF
df['final_text'] = df['tokens'].apply(lambda x: ' '.join(x))

X = tfidf.fit_transform(df['final_text'])
y = df['sentiment'].map({'positive': 1, 'negative': 0})  # adjust labels if different

print("Shape of TF-IDF matrix:", X.shape)

Shape of TF-IDF matrix: (50000, 10000)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (40000, 10000)
Test shape: (10000, 10000)


In [10]:
model = MultinomialNB(alpha=1.0)  # alpha=1.0 applies Laplace smoothing
model.fit(X_train, y_train)

MultinomialNB()

In [11]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

Accuracy: 0.8707

Confusion Matrix:
[[4282  718]
 [ 575 4425]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.88      0.86      0.87      5000
    Positive       0.86      0.89      0.87      5000

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



In [13]:
from nltk.corpus import wordnet # Added for lemmatization

def get_wordnet_pos(word):
    """Map NLTK position tags to WordNet tags"""
    # pos_tag expects a list of words, and returns a list of (word, tag) tuples
    tag = pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB,
                "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN) # Default to Noun if not found

def lemmatize_tokens(tokens):
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in tokens]
    return lemmatized_tokens

def predict_sentiment(text):
    cleaned = clean_text(text)
    tokens = tokenize_and_remove_stopwords(cleaned)
    final = lemmatize_tokens(tokens)
    # The tfidf vectorizer expects a single string, so join the lemmatized tokens
    vectorized = tfidf.transform([' '.join(final)])
    prediction = model.predict(vectorized)[0]
    return "Positive" if prediction == 1 else "Negative"

test_sentences = [
    "This product is absolutely amazing, I love it!",
    "This is terrible, I want a refund immediately.",
    "I am not happy with this purchase at all.",
    "Not bad, actually pretty good for the price."
]

for sentence in test_sentences:
    print(f"'{sentence}' → {predict_sentiment(sentence)}")

'This product is absolutely amazing, I love it!' → Negative
'This is terrible, I want a refund immediately.' → Negative
'I am not happy with this purchase at all.' → Positive
'Not bad, actually pretty good for the price.' → Negative
